# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. (1, 2, 3), (0, 1, 1)
2. (4, 4, 4) столбцом
3. [[0 1 2]
 [1 2 3]
 [2 3 4]]

4. 6
5. 7.5

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:
# 1
print(clean.sum())
oct_revenue = amount[clean & (month == "2026-10")].sum()
sep_revenue = amount[clean & (month == "2026-09")].sum()
print(oct_revenue)
print(sep_revenue)
print((oct_revenue - sep_revenue) / sep_revenue * 100) # Данные стали чище

4739
34419520.0
35243410.0
-2.337713632137185


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:
# 2
cats = np.sort(np.unique(cat))
months = np.sort(np.unique(month))
cat_month_rev = np.zeros((len(cats), len(months)))
for i, c in enumerate(cats):
    for j, m in enumerate(months):
        cat_month_rev[i, j] = amount[clean & (cat == c) & (month == m)].sum()

print("Категории x Месяцы:\n", cat_month_rev)
total_cat = cat_month_rev.sum(axis=1)
print("Больше всего выручки:", cats[total_cat.argmax()], f"({total_cat.max():,.2f} ₸)")
cat_chg = (cat_month_rev[:, 1] - cat_month_rev[:, 0]) / cat_month_rev[:, 0] * 100
print("Сильнее всего упала:", cats[cat_chg.argmin()], f"({cat_chg.min():.2f}%)")

Категории x Месяцы:
 [[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше всего выручки: Электроника (34,370,430.00 ₸)
Сильнее всего упала: Одежда (-7.21%)


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [6]:
# 3
cat_counts = np.zeros((len(cats), len(months)), dtype=int)
for i, c in enumerate(cats):
    for j, m in enumerate(months):
        cat_counts[i, j] = (clean & (cat == c) & (month == m)).sum()

cat_month_avg = cat_month_rev / cat_counts
print("Средний чек в октябре: ", cat_month_avg[:, 1])
print("Самый высокий средний чек в октябре:", cats[cat_month_avg[:, 1].argmax()])

Средний чек в октябре:  [10768.46638655  3815.75250836  6158.59770115 13045.59815951
 33600.1980198 ]
Самый высокий средний чек в октябре: Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [7]:
# 4
cities_sorted = np.sort(np.unique(city))
city_month_rev = np.zeros((len(cities_sorted), len(months)))

for i, c in enumerate(cities_sorted):
    for j, m in enumerate(months):
        city_month_rev[i, j] = amount[clean & (city == c) & (month == m)].sum()

table_shares = city_month_rev / city_month_rev.sum(axis=0)
print("Субъекты по столбцам сумм:", table_shares.sum(axis=0))

shym_mask = cities_sorted == "Шымкент"
shym_idx = np.where(cities_sorted == 'Шымкент')[0][0]
print(f"Доля Шымкента: сентябрь {table_shares[shym_idx, 0]:.4f} -> октябрь {table_shares[shym_idx, 1]:.4f}")

Субъекты по столбцам сумм: [1. 1.]
Доля Шымкента: сентябрь 0.1293 -> октябрь 0.1175


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [8]:
# 5
total_change = oct_revenue - sep_revenue
city_change = city_month_rev[:, 1] - city_month_rev[:, 0]
contributions = city_change / total_change * 100
for c, contr in zip(cities_sorted, contributions):
    print(f"{c}: {contr:.2f}% от общего падения")

# Фраза для менеджера: «Главную причину падения выручки нужно искать в Шымкенте, так как именно на этот город приходится 62.5% общего снижения продаж.»

Актобе: 4.44% от общего падения
Алматы: -17.23% от общего падения
Астана: 22.30% от общего падения
Атырау: -5.57% от общего падения
Караганда: 33.60% от общего падения
Шымкент: 62.46% от общего падения


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [13]:
# 6
print(table_shares)
print(table_shares.sum(axis=1))
share = table_shares / table_shares.sum(axis=1, keepdims=True)
print(share)
# AI попытался разделить матрицу (6, 2) на вектор (6,).
# В NumPy при делении двумерного массива на одномерный размерности по строкам не совпадают.
# Нужно использовать keepdims=True у функции sum(axis=1)

[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]
[0.14692645 0.83688257 0.51347479 0.054395   0.20148184 0.24683935]
[[0.49765659 0.50234341]
 [0.49165176 0.50834824]
 [0.49922188 0.50077812]
 [0.48198151 0.51801849]
 [0.51381042 0.48618958]
 [0.52401527 0.47598473]]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8